# CAM（CIFAR100を用いたCNNの可視化）

---
## 目的
Class Activation Mapping (CAM) の仕組みを理解する．CAMを用いて，CIFAR100データセットに対するネットワークの判断根拠（画像のどの領域を根拠にクラスを予測したか）を可視化する．

## Class Activation Mapping (CAM)
CAM[1]は，畳み込み層とGlobal Average Pooling (GAP)，全結合層（1層のみ）から構成されるネットワークに対して，追加の学習や構造変更なしに適用できる可視化手法です．

CNNは，畳み込み層を通すことで，入力画像のどの位置に何の特徴があるかという空間的な情報を保持した特徴マップを獲得します．GAPは，この特徴マップをチャネルごとに空間方向で平均し，1つの値（特徴ベクトル）に変換します．最後の全結合層は，このGAP後の特徴ベクトルに対して重み付き和を取ることでクラスごとのスコアを計算します．

つまり，クラス$c$のスコアは，GAP前の特徴マップの各チャネル$k$を，全結合層の重み$w_{c,k}$で重み付けして足し合わせたものと等価です．CAMは，この「重み付け」をGAPで平均する前の特徴マップ（空間情報を保持したまま）に対して行うことで，クラス$c$と判定する際にどの空間位置が貢献したかを可視化します．

$$M_c(x, y) = \sum_{k} w_{c,k} \, f_k(x, y)$$

ここで$f_k(x, y)$は最後の畳み込み層が出力する特徴マップのチャネル$k$，位置$(x, y)$における値です．

[1] B. Zhou et al., "Learning Deep Features for Discriminative Localization," CVPR, 2016.

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchsummary
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．データセットの詳細については`classification/resnet.ipynb`を参照してください．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)
class_names = train_data.classes

print(len(train_data), len(test_data))

## ネットワークの定義
CAMは「畳み込み層 → GAP → 全結合層（1層）」という構造を前提とする手法です．そこで，`classification/resnet.ipynb`と全く同じCIFAR版ResNet（BasicBlock, depth=20）を使用します．このネットワークは，3つのResidual Blockの後に`avgpool`（GAP）と`fc`（全結合層1層）が続く構造になっており，CAMをそのまま適用できます．ネットワークの詳細（Residual Blockの構造など）は`classification/resnet.ipynb`を参照してください．

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.convs(x)
        out += residual
        return self.relu(out)


class ResNetBasicBlock(nn.Module):
    def __init__(self, depth, n_class=100):
        super().__init__()
        assert (depth - 2) % 6 == 0, 'depth should be 6n+2 (e.g. 20, 32, 44).'
        n_blocks = (depth - 2) // 6

        self.inplanes = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)  # CAMで使う最後の畳み込み特徴マップ（8x8x64）

        self.avgpool = nn.AvgPool2d(8)  # Global Average Pooling
        self.fc = nn.Linear(64 * BasicBlock.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * BasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * BasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * BasicBlock.expansion),
            )
        layers = [BasicBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * BasicBlock.expansion
        for _ in range(n_blocks - 1):
            layers.append(BasicBlock(self.inplanes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        feature_map = self.layer3(x)  # GAP前の特徴マップ (N, 64, 8, 8)

        x = self.avgpool(feature_map)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x, feature_map

## ネットワークの作成
ネットワークを作成します．最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします（`resnet.ipynb`と同じ設定）．

In [ ]:
model = ResNetBasicBlock(depth=20, n_class=100).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
CIFAR100データセットを用いて学習します．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss().to(device)

model.train()
train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y, _ = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print(f'epoch: {epoch}, mean loss: {sum_loss / len(train_loader):.4f}, mean accuracy: {count.item() / len(train_data):.4f}, elapsed_time: {time() - train_start:.4f}')

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()
count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y, _ = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## CAMの計算
学習済みのネットワークを用いてCAMを計算する関数を実装します．`forward`が返す`feature_map`（GAP前，(N, 64, 8, 8)）と，`fc`層の重み`fc.weight`（(100, 64)）を用いて，指定したクラス$c$に対するCAMを計算します．計算したCAMは，入力画像と同じサイズにバイリニア補間で拡大し，画像に重ねて表示できるようにします．

In [ ]:
def compute_cam(feature_map, fc_weight, class_idx, out_size=32):
    """feature_map: (1, C, H, W), fc_weight: (n_class, C) から，class_idxに対するCAMを計算する"""
    weight_c = fc_weight[class_idx]  # (C,)
    cam = torch.einsum('c,chw->hw', weight_c, feature_map[0])  # チャネル方向に重み付け和
    cam = F.relu(cam)  # 負の寄与は0にする（正の根拠のみ可視化する）
    cam = cam / (cam.max() + 1e-8)  # 0〜1に正規化
    cam = F.interpolate(cam.view(1, 1, *cam.shape), size=out_size, mode='bilinear', align_corners=False)
    return cam.squeeze().cpu().numpy()


def overlay_cam(image, cam, alpha=0.5):
    """image: (H, W, 3) の0〜1のnumpy配列，cam: (H, W) の0〜1のnumpy配列にjetカラーマップを重ねる"""
    heatmap = plt.get_cmap('jet')(cam)[:, :, :3]  # (H, W, 3)，RGBAのうちRGBのみ使用
    return (1 - alpha) * image + alpha * heatmap

## 可視化
評価用データからいくつかの画像を取り出し，予測クラスに対するCAMを計算して可視化します．赤色に近いほど，ネットワークがそのクラスと判断する根拠として強く着目した領域であることを示します．

In [ ]:
model.eval()
n_show = 6
fig, axes = plt.subplots(2, n_show, figsize=(2.2 * n_show, 4.6))

for i in range(n_show):
    image, label = test_data[i]
    image_batch = image.unsqueeze(0).to(device)

    with torch.no_grad():
        y, feature_map = model(image_batch)
        pred_class = torch.argmax(y, dim=1).item()
        cam = compute_cam(feature_map, model.fc.weight, pred_class)  # fc.weightは勾配を持つパラメータのため，no_grad内で扱う

    image_np = image.permute(1, 2, 0).numpy()
    overlay = overlay_cam(image_np, cam)

    axes[0, i].imshow(image_np); axes[0, i].axis('off')
    axes[0, i].set_title(f'GT: {class_names[label]}', fontsize=9)
    axes[1, i].imshow(overlay); axes[1, i].axis('off')
    axes[1, i].set_title(f'pred: {class_names[pred_class]}', fontsize=9)

plt.tight_layout()
plt.show()

## 課題

1. 予測クラス（`pred_class`）ではなく，正解クラス（`label`）や別の任意のクラスを指定してCAMを計算し，予測が誤っている画像に対してどのような違いが現れるか確認してください．
2. 誤分類された画像を集め，それらのCAMを可視化して，ネットワークがどのような領域に着目して誤った判断をしているか考察してください．
3. `layer3`ではなく`layer2`の出力（GAPを通さない中間特徴マップ）を使ってCAMを計算しようとした場合，どのような変更が必要になるか（あるいは，なぜそのままでは計算できないか）を考えてください．